# 05 — Caged CrafText random policy trajectory
Run a mask-aware random policy, inspect observations and save a replay artifact.


In [ ]:
from pathlib import Path
import json
import jax
from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.rollouts.random_policy import sample_masked_actions
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.tasks import CrafTextTaskSampler


In [ ]:
config = load_mvp_config(Path('configs/benchmarks/caged_craftext_full.yaml'))
runtime = build_craftext_runtime(config)
task_sampler = CrafTextTaskSampler.from_runtime(
    runtime,
    horizon=config.environment.horizon,
    mode='fixed',
    fixed_instruction_index=config.environment.instruction_index,
)
prepared_goal, prepared_instruction_index = task_sampler.task_at(config.environment.instruction_index)
reset = runtime.adapter.reset(jax.random.PRNGKey(config.run.seed))
print('observation shape:', reset.observation.shape)
print('action count:', runtime.action_count)
print('prepared CrafText task:', prepared_instruction_index, prepared_goal)


In [ ]:
state, mask = reset.state, reset.action_mask
trajectory = []
for step in range(config.environment.horizon):
    key_action, key_env = jax.random.split(jax.random.PRNGKey(config.run.seed + step + 1))
    action = int(sample_masked_actions(key_action[None, :], mask[None, :])[0])
    transition = runtime.adapter.step(key_env, state, action)
    trajectory.append({'step': step, 'action': action, 'reward': float(transition.reward), 'terminated': bool(transition.terminated)})
    state, mask = transition.state, transition.action_mask
    if bool(transition.terminated): break
trajectory


In [ ]:
output = Path('artifacts/trajectories/caged-random-policy.json')
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps({'config': 'configs/benchmarks/caged_craftext_full.yaml', 'steps': trajectory}, indent=2))
print(output)
